# 🛠️ 实验〇：环境安装指南

**MobileNetV3 图像分类实战教程**

---

## 学习目标

1. 安装 Python 和必要的开发工具
2. 安装 PyTorch 深度学习框架
3. 安装 Jupyter Notebook 并启动
4. 验证环境是否配置正确
5. 了解 NPU / CPU 环境差异

---

> ⏱️ 预计时间：20-30 分钟
> 
> 💻 推荐配置：Ascend NPU（可选，没有 NPU 也能运行，训练会慢一些）

---
## 1. 安装 Python

### 方案 A：使用 Anaconda（推荐）

Anaconda 包含了 Python、Jupyter 和常用的科学计算库，适合教学场景。

1. 访问 [Anaconda 官网](https://www.anaconda.com/download) 下载安装包
2. 根据操作系统选择对应版本（Windows / macOS / Linux）
3. 安装完成后，打开终端（Terminal）验证：

```bash
conda --version
python --version
```

### 方案 B：使用系统 Python（轻量）

如果不想安装 Anaconda，系统自带的 Python 也可以：

```bash
# Ubuntu / Debian
sudo apt update
sudo apt install python3 python3-pip python3-venv -y

# macOS（使用 Homebrew）
brew install python

# Windows
# 从 https://www.python.org/downloads/ 下载安装
```

**要求：Python ≥ 3.8**

In [ ]:
# ====== 1. 检查 Python 版本 ======
import sys
print(f"Python 版本: {sys.version}")

major, minor = sys.version_info.major, sys.version_info.minor
if major >= 3 and minor >= 8:
    print("✅ Python 版本满足要求 (≥ 3.8)")
else:
    print("❌ Python 版本过低，请升级到 3.8 以上")

---
## 2. 安装 PyTorch 与 torch-npu

在华为昇腾 NPU 平台上，需要安装 PyTorch 和 torch-npu 适配插件。

### 2.1 检查 NPU 设备

```bash
# 在终端中运行
npu-smi info
```

如果显示 NPU 信息（如 Ascend 910B），说明 NPU 驱动正常。
如果显示 `command not found`，需要先安装 CANN 工具包。

### 2.2 安装命令

PyTorch、torch-npu 和 CANN 软件包与硬件型号强相关，安装前建议查看
[昇腾官方版本适配文档](https://www.hiascend.com/document/detail/zh/Pytorch/2600/releasenote/docs/zh/release_notes/release_notes.md)。

```bash
# 安装 NPU 版本的 PyTorch + torch-npu
pip install torch torch-npu
```

> 💡 **提示**：如果使用华为云 ModelArts 或昇腾官方镜像，环境通常已预装。
> 如果下载速度慢，可以添加国内镜像源：
> ```bash
> pip install torch torch-npu -i https://pypi.tuna.tsinghua.edu.cn/simple
> ```


In [ ]:
# ====== 2. 检查 PyTorch 安装 ======
try:
    import torch
    print(f"✅ PyTorch 已安装")
    print(f"   PyTorch 版本: {torch.__version__}")
    
    import torchvision
    print(f"✅ Torchvision 已安装")
    print(f"   Torchvision 版本: {torchvision.__version__}")
    
except ImportError as e:
    print(f"❌ 安装缺失: {e}")
    print("请运行: pip install torch torchvision")

In [ ]:
# ====== 3. 检查 PyTorch 与 NPU 可用性 ======
import torch

try:
    import torch_npu
    print("✅ torch-npu 已安装")
except ImportError:
    print("⚠️  torch-npu 未安装，将使用 CPU 模式")

print("=" * 50)
print("Ascend NPU 环境检测")
print("=" * 50)

npu_available = torch.npu.is_available() if hasattr(torch, 'npu') else False
print(f"NPU 可用: {npu_available}")

if npu_available:
    print(f"NPU 数量: {torch.npu.device_count()}")
    for i in range(torch.npu.device_count()):
        print(f"  NPU {i}: {torch.npu.get_device_name(i)}")
    print("\n🎉 可以使用 NPU 加速训练！速度会比 CPU 快 10-50 倍。")
else:
    print("\n⚠️  未检测到 NPU")
    print("   - 本教程使用 CPU 也可以运行，但训练速度会慢很多")
    print("   - 建议在华为昇腾平台上运行本教程以获得最佳体验")


---
## 3. Ascend NPU 开发环境说明

### 3.1 torch-npu 简介

`torch-npu` 是 PyTorch 的 Ascend NPU 适配插件，提供了与 CUDA 几乎一致的 API。
只需 `import torch_npu` 即可使用 `torch.npu.*` 接口。

### 3.2 关键 API 对照

<table style="margin-left: 0; margin-right: auto; border-collapse: collapse; border: 1px solid #ddd;">
  <thead>
    <tr style="background-color: #f2f2f2;">
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">操作</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">CUDA（NVIDIA GPU）</th>
      <th style="border: 1px solid #ddd; padding: 8px; text-align: left;">torch-npu（昇腾 NPU）</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">导入</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><code>import torch</code></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><code>import torch; import torch_npu</code></td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">设备指定</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><code>torch.device('cuda')</code></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><code>torch.device('npu:0')</code></td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">可用性</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><code>torch.cuda.is_available()</code></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><code>torch.npu.is_available()</code></td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">数据迁移</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><code>.to('cuda')</code></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><code>.to('npu:0')</code></td>
    </tr>
    <tr>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;">同步</td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><code>torch.cuda.synchronize()</code></td>
      <td style="border: 1px solid #ddd; padding: 8px; text-align: left;"><code>torch.npu.synchronize()</code></td>
    </tr>
  </tbody>
</table>

### 3.3 确认环境

运行下方代码单元格，确认 NPU 环境正常工作。


In [ ]:
# ====== 4. 确认 NPU 可用性 ======
import torch

# 导入 torch_npu 是使用 NPU 的关键一步
import torch_npu

print("=" * 50)
print("Ascend NPU 环境确认")
print("=" * 50)

npu_available = torch.npu.is_available()
print(f"NPU 可用: {npu_available}")

if npu_available:
    print(f"NPU 数量: {torch.npu.device_count()}")
    for i in range(torch.npu.device_count()):
        print(f"  NPU {i}: {torch.npu.get_device_name(i)}")
    print("\n🎉 NPU 环境就绪，可以开始实验！")
else:
    print("\n❌ NPU 不可用，请检查:")
    print("   1. 是否已安装 torch-npu")
    print("   2. CANN 版本是否匹配")
    print("   3. 运行 npu-smi info 确认 NPU 驱动正常")


---
## 4. 安装其他依赖

本项目还需要以下 Python 库：

In [ ]:
# ====== 5. 安装项目依赖 ======
# 如果以下包缺失，取消注释并运行：
# !pip install torch torchvision torch-npu Pillow tqdm matplotlib scipy

# 检查依赖是否已安装
import importlib

required_packages = [
    ('PIL', 'Pillow', '图像处理'),
    ('tqdm', 'tqdm', '进度条'),
    ('matplotlib', 'matplotlib', '绘图'),
    ('scipy', 'scipy', '科学计算'),
]

print("检查项目依赖...")
all_ok = True
for import_name, pkg_name, desc in required_packages:
    try:
        importlib.import_module(import_name)
        print(f"  ✅ {pkg_name:15s} - {desc}")
    except ImportError:
        print(f"  ❌ {pkg_name:15s} - {desc} (缺失)")
        all_ok = False

if all_ok:
    print("\n🎉 所有依赖已满足！")
else:
    print("\n⚠️  部分依赖缺失，请运行:")
    print("   pip install Pillow tqdm matplotlib scipy")

---
## 5. 安装 Jupyter Notebook

如果你使用的是 **Anaconda**，Jupyter 已经预装好了。

如果使用的是系统 Python，需要手动安装：

In [ ]:
# ====== 5. 检查 Jupyter ======
try:
    import notebook
    print(f"✅ Jupyter Notebook 已安装")
    print(f"   版本: {notebook.__version__}")
except ImportError:
    print("❌ Jupyter Notebook 未安装")
    print("运行以下命令安装:")
    print("   pip install jupyter notebook")

---
## 6. 启动 Jupyter Notebook

安装完成后，在**项目根目录**启动 Jupyter：

```bash
# 进入项目目录
cd MobileNetV3-Pytorch-master

# 启动 Jupyter Notebook
jupyter notebook

# 或者使用 Jupyter Lab（界面更现代）
jupyter lab
```

启动后，浏览器会自动打开 Jupyter 界面。点击 `notebooks/` 文件夹，然后按顺序打开教程即可。

### 在服务器上启动（远程访问）

```bash
# 在服务器上启动，监听所有 IP
jupyter notebook --ip=0.0.0.0 --port=8888 --no-browser

# 设置密码（第一次访问时）
jupyter notebook password
```

然后在本地浏览器访问：`http://服务器IP:8888`

---
## 7. 在非 NPU 环境运行（备选）

如果不在昇腾 NPU 平台上，本教程也可以在 CPU 上运行（速度较慢）。
也可以在 Google Colab 等 GPU 平台上运行，但需要将设备设置为 `cuda`。

```python
# 在 Colab 中运行（需要手动修改设备）
from google.colab import drive
drive.mount('/content/drive')

# 或者直接从 GitHub 拉取代码
!git clone https://github.com/你的用户名/MobileNetV3-Pytorch-master.git
%cd MobileNetV3-Pytorch-master

# 安装依赖
!pip install torch torchvision Pillow tqdm matplotlib scipy
```

---
## 8. 验证项目文件完整性

确保项目文件都正确到位。

In [ ]:
# ====== 7. 验证项目文件 ======
import os

required_files = [
    '../model.py',
    '../main.py',
    '../preprocess.py',
    '../inference.py',
]

print("检查项目文件完整性...")
all_ok = True
for f in required_files:
    path = os.path.abspath(f)
    exists = os.path.exists(path)
    if exists:
        size = os.path.getsize(path) if os.path.isfile(path) else 'dir'
        print(f"  ✅ {f:45s} ({size})")
    else:
        print(f"  ❌ {f:45s} (缺失)")
        all_ok = False

if all_ok:
    print("\n🎉 项目文件完整，可以开始学习了！")
else:
    print("\n⚠️  部分文件缺失，请检查项目是否完整下载")

# 检查数据集
data_dir = os.path.abspath('../data/tiny-imagenet-200')
if os.path.exists(data_dir):
    train_dirs = len([d for d in os.listdir(os.path.join(data_dir, 'train')) 
                      if os.path.isdir(os.path.join(data_dir, 'train', d))]) if os.path.exists(os.path.join(data_dir, 'train')) else 0
    val_dirs = len([d for d in os.listdir(os.path.join(data_dir, 'val')) 
                    if os.path.isdir(os.path.join(data_dir, 'val', d))]) if os.path.exists(os.path.join(data_dir, 'val')) else 0
    print(f"\n数据集: {train_dirs} 个训练类, {val_dirs} 个验证类")

---
## 9. 常见问题排查

### ❌ `pip` 命令找不到
```bash
# Ubuntu/Debian
sudo apt install python3-pip

# 或者使用 pip3
pip3 install ...
```

### ❌ 安装 PyTorch 速度很慢
```bash
# 使用国内镜像（清华大学 TUNA）
pip install torch torch-npu -i https://pypi.tuna.tsinghua.edu.cn/simple
```

### ❌ Ascend NPU 检测不到
```bash
# 确认 CANN 已正确安装
npu-smi info

# 确认 torch-npu 版本与 CANN 匹配
pip show torch-npu
```

### ❌ torch.npu.is_available() 返回 False
```bash
# 1. 检查是否有 NPU 设备
npu-smi info
# 2. 检查当前用户是否有权限访问 NPU
# 3. 检查 PyTorch 和 torch-npu 版本是否匹配
# 4. 尝试重新安装 torch-npu
pip uninstall torch-npu -y
pip install torch torch-npu
```

### ❌ Jupyter Notebook 无法启动
```bash
# 查看错误信息
jupyter notebook --debug

# 重置配置
jupyter notebook --generate-config
```

### ❌ 内存不足
```bash
# 训练时减小 batch size
# 在代码中设置 BATCH_SIZE = 32 或更小
```

---
## ✅ 环境准备确认清单

运行完上面的检查后，确认以下项目全部打勾：

- [ ] Python ≥ 3.8
- [ ] PyTorch 已安装
- [ ] torch-npu 已安装
- [ ] Torchvision 已安装
- [ ] 项目依赖已安装（Pillow, tqdm, matplotlib, scipy）
- [ ] Jupyter Notebook 已安装
- [ ] NPU 可用（`torch.npu.is_available() == True`）
- [ ] 项目文件完整
- [ ] Tiny ImageNet 数据集已就绪

全部完成后，就可以开始第一个实验了！

---


## 课后练习

1. (单选题) 在 NPU 推理环境中，以下哪段导入与初始化顺序是正确的？
   - A. 先 import torch_npu，再 import torch
   - B. 先 import torch，再 import torch_npu，并执行一次 torch.npu.set_device(0)
   - C. 只 import torch_npu，无需 import torch
   - D. 先 import torch.cuda，再 import torch_npu

2. (单选题) npu-smi 显示 8 张 NPU 正常，但 torch.npu.is_available() 返回 False，最可能的原因是？
   - A. torch/CANN/torch_npu 版本不匹配或驱动未正确加载
   - B. 数据集路径错误
   - C. 模型未定义
   - D. 学习率过大

3. (多选题) 以下哪些检查组合能较完整地验证 CANN 与 torch_npu 环境？
   - A. npu-smi info 查看设备状态
   - B. torch.npu.is_available() 与 torch.npu.device_count()
   - C. 打印 torch.__version__ 与 torch_npu.__version__
   - D. 创建 torch.ones(4, device="npu:0") 并回拷到 CPU

4. (多选题) Docker 容器共享内存过小可能引发哪些现象？
   - A. DataLoader 多进程 worker 报共享内存相关 OSError
   - B. /dev/shm 写入失败
   - C. 模型权重文件损坏
   - D. 数据加载速度异常下降

5. (判断题) 先导入 torch 再导入 torch_npu，是 torch_npu 正确使用的必要条件之一。

6. (判断题) 在只有 NPU 的机器上，torch.cuda.is_available() 返回 True，因此可以直接沿用 CUDA 设备选择逻辑。

7. (填空题) 在 CANNLab 环境中，查看 NPU 设备数量、健康状态与显存占用应使用命令 ____。

8. (填空题) 限制当前进程可见 NPU 卡的环境变量是 ____。

9. (简答题) 为什么推荐使用 conda 为 CANN/PyTorch 实验创建独立环境？请从依赖隔离、版本锁定和复现三个角度说明。

10. (简答题) 安装或升级 CANN、驱动或 torch_npu 后，为什么必须重启 Jupyter kernel 才能生效？

11. (代码设计题) 编写 get_device()，要求优先返回 npu:0，不可用时回退 cpu，并打印 torch、torch_npu 版本与 NPU 数量。

12. (单选题) 在 NPU 上执行 torch.npu.synchronize() 的主要目的是？
   - A. 释放显存
   - B. 等待异步 kernel 全部完成，保证后续计时与读取结果可靠
   - C. 触发图编译
   - D. 清空缓存

13. (多选题) 日志出现 Permission mismatch: The owner of ... does not match 时，常见原因与处理包括？
   - A. 包由 root 安装而当前用户非 root
   - B. 文件 owner 与运行用户不一致
   - C. 通过 chown 或重建虚拟环境缓解
   - D. 直接删除该文件即可彻底解决

14. (判断题) CANNLab 镜像预装了 torch/torch_npu，因此实验中不需要再安装任何其他 Python 依赖。

15. (简答题) DataLoader 设置 num_workers=8 后容器报共享内存不足，请给出至少三种排查或解决思路。

> 参考答案见 answer/02.02_environment_setup_answer.ipynb。